# VS Code Git Workflow Training
### 40-Minute Hands-On Session | AWA Windows VM + VS Code

Welcome! This notebook is your interactive guide for today's session.

**How you got here:** You downloaded this repository as a ZIP from GitHub,
and opened the folder in VS Code. The notebook is ready to run with no cloning needed.

**Before we start:** confirm Git Bash is your VS Code integrated terminal.
Go to **Terminal → New Terminal** and verify the prompt shows a `$` (not `>`).
If it shows `>`, go to **File → Preferences → Settings**, search for
`terminal.integrated.defaultProfile.windows`, and set it to **Git Bash**.

---
## 🔒 Organisation Security Policy: Read Before You Start

> **This section is mandatory. Violations may result in a data breach, disciplinary action, or both.**

As a member of the **RBK-D-IT** GitHub organisation you have access to private repositories
that contain proprietary data, code, and business logic.
The actions you learn today are powerful. A small mistake can permanently expose
confidential information to the public internet.
Read the five rules below before touching any command.

---

### Rule 1: Never change a repository's visibility to Public

> ⚠️ **SECURITY: Repo visibility:**
> GitHub's repository Settings page contains a **Danger Zone** section with a
> **"Change repository visibility"** button.
> Only an **organisation administrator** may use this on an org repository.
> If you click it on a private repo, **every commit, file, and piece of history
> becomes publicly readable on the internet immediately**.
> There is no undo. Even after switching back to private, search engines and scrapers
> may have already indexed the content.

**What to do instead:** If a project needs to be made public, raise a request with
your IT lead. Never act on the Danger Zone yourself.

---

### Rule 2: Never fork a private org repo to your personal GitHub account

> ⚠️ **SECURITY: Forking private repos:**
> Forking copies the **entire commit history** of a repository into your personal GitHub account.
> If you later make that personal fork public, or if your personal account has less-restricted
> settings, the org's confidential code becomes publicly accessible.
> Git history cannot be scrubbed after exposure; the only remediation is a full security incident response.

**What to do instead:** Work on branches within the org repository.
The fork workflow in Section 5 of this training uses a **public** practice repo
(`git-vscode-practice`), so forking it is safe.
Do **not** apply the fork workflow to any private org repo without explicit approval from your team lead.

---

### Rule 3: Never re-publish cloned org content in a personal public repository

> ⚠️ **SECURITY: Re-publishing clones:**
> When you `git clone` a repository you download every file **and every commit** in its history.
> If you then push those files or that history to a public personal repository
> by adding a new remote pointing to your own public repo and pushing,
> you have effectively published the org's private repository.
> This constitutes a data breach regardless of intent.

**What to do instead:** Keep cloned org repos on the AWA VM only.
Do not transfer them to personal machines, personal cloud drives
(OneDrive personal, Google Drive, Dropbox), or any public-facing storage.

---

### Rule 4: Treat your Personal Access Token like a password

> ⚠️ **SECURITY: PAT exposure:**
> A PAT with `repo` scope grants whoever holds it **read and write access to every private
> repository** you have access to, including org repos.
> If you accidentally paste a PAT into a file and commit it, or share it in a message,
> revoke it immediately at **GitHub → Settings → Developer settings → Personal access tokens**
> and generate a fresh one. Report the incident to your IT security team.

**What to do instead:** Let Git Credential Manager store the token (Section 2).
Never type or paste your PAT into a script, notebook cell, or chat message.

---

### Rule 5: Keep org repo clones on the AWA VM

> ⚠️ **SECURITY: Machine boundary:**
> The AWA VM is an organisation-managed, encrypted environment. Your personal laptop or home PC is not.
> Do not copy, zip, or otherwise transfer a cloned org repository outside the AWA boundary
> without explicit approval from your data governance team.

---

**If you are ever unsure whether an action is safe: stop and ask your team lead or IT before proceeding.**

---

## Section 1: Introduction & Welcome (2 min)

You completed the **GitHub Foundations Skills Track** on DataCamp. Great work!
That track taught you Git concepts in a browser-based sandbox.
Today, we move those same commands into your real AWA data VM using **VS Code**.

This notebook covers three workflows every data analyst uses daily:
- Cloning an existing repository from GitHub.
- Initialising a local project and tracking it with Git.
- Branching, pushing changes, and opening a Pull Request.

Each action is shown three ways so you can find the style that suits you.

In [ ]:
import sys

print(f"Python {sys.version}")
print("Environment OK — let's go.")

## Section 2: Initial Setup & Authentication (5 min)

Before Git can sign your commits or talk to GitHub, two things must be configured:
your **identity** (name + email) and your **authentication method**.

### Check Git Version

In [ ]:
!git --version

### Configure Your Global Identity

Edit the two values below, then run the cell.
This writes to `~/.gitconfig` and persists across all projects on this machine.

In [ ]:
!git config --global user.name "Your Name"
!git config --global user.email "your.email@example.com"
!git config --global init.defaultBranch main
!git config --list --global

### Authentication: Two Methods

**Method 1 (VS Code UI): Sign in with GitHub OAuth**

> 👥 **EVERYONE:** Click the **Accounts** icon (person silhouette) in the
> bottom-left Activity Bar. Select **Sign in with GitHub**. Complete the browser
> OAuth flow. VS Code stores the token and uses it for all GitHub operations
> (Clone, Push, Pull Requests) without touching the terminal.

This method covers all VS Code UI actions. Complete Method 2 as well so that
`git push` from the terminal also works.

**Method 2 (Terminal): Personal Access Token via Git Credential Manager**

Git for Windows ships with **Git Credential Manager (GCM)**, which stores your PAT
securely in the Windows Credential Store. You only need to do this once.

Step 1: On GitHub.com go to **Settings → Developer settings → Personal access tokens → Tokens (classic)**.

> ⚠️ **SECURITY: PAT scope:**
> The `repo` scope gives the token **full read/write access to every private repository**
> you can access, including all private org repos.
> Use the narrowest scope your work actually requires.
> If you are only reading and pushing code, consider a **fine-grained personal access token**
> (GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens)
> scoped to specific repositories instead.
> **Never paste the token value into any file, cell, or message.** GCM stores it for you.

Step 2: Click **Generate new token**. For this training, check `repo` and `workflow`. The `repo` scope lets you clone and push to the practice repo, `workflow` covers GitHub Actions. In your real day-to-day work, prefer a fine-grained token scoped to only the repos you need. Copy the token once generated.
Step 3: Run the cell below. GCM will prompt for your PAT on the next network operation.

In [ ]:
!git config --global credential.helper manager

## Section 3: Remote-First Workflow: Cloning (8 min)

Cloning downloads a full copy of a GitHub repository to your local machine,
including all branches and history.

> ⚠️ **SECURITY: Cloning private repos:**
> Cloning creates a **complete local copy** of the repository, including its entire commit history.
> If the repo you are cloning is a **private org repository**, treat the resulting local folder
> as confidential:
> - Do **not** push it (or any part of it) to a personal public repository.
> - Do **not** copy the folder to a personal machine or personal cloud drive.
> - Do **not** add a remote pointing to a public repo and push, as this publishes the org's full history.
>
> This training clones a **public** practice repo, so the steps below are safe.
> Apply this caution whenever you clone any private org repository in your real work.


We will clone the shared practice repository into your **Downloads** folder:
**https://github.com/RBK-D-IT/git-vscode-practice**

> **Note:** Methods 1 & 2 (VS Code UI) require a **new VS Code window** because this window
> is open on the extracted ZIP folder (not a Git repo), so the Source Control sidebar
> shows **"Initialize Repository"** rather than "Clone Repository".
> **Method 3 (terminal) works from this window**, so run the cell below.

### Method 1: VS Code UI (Source Control Tab)

> 👥 **EVERYONE:** Open a **new VS Code window**: **File → New Window** (or **Ctrl+Shift+N**).
> In the new empty window, open the **Source Control** tab (**Ctrl+Shift+G**).
> You will see **Clone Repository** at the top. VS Code shows this whenever no folder is open.
> Click **Clone Repository**. Paste:
> `https://github.com/RBK-D-IT/git-vscode-practice`
> Choose your **Downloads** folder as the destination when prompted.
> Click **Open** when VS Code asks to open the cloned repository.

### Method 2: Command Palette (Power-User)

> 👥 **EVERYONE:** In the **new VS Code window**, press **Ctrl+Shift+P**.
> Type `Git: Clone` and press Enter.
> Paste the repo URL. Choose your **Downloads** folder as the destination.

### Method 3: Terminal

Run the cell below from this notebook window.

In [ ]:
!git -C ~/Downloads clone https://github.com/RBK-D-IT/git-vscode-practice.git

## Section 4: Local-First Workflow: Init, Stage, Commit (10 min)

Sometimes you start a project locally before it exists on GitHub.
`git init` turns any folder into a Git repository.

We will:
1. Create a new folder `my-local-project` inside your **Downloads** folder.
2. Add two files: `hello_world.py` and `notes.md`.
3. Stage and commit them (three ways).

### Create the project folder and files

#### Method 1: VS Code Explorer (Interactive)

> 👥 **EVERYONE:** Open the **Explorer** panel (**Ctrl+Shift+E**).
> In the sidebar, right-click inside the **Downloads** folder → **New Folder** →
> type `my-local-project` → press **Enter**.
>
> Right-click on `my-local-project` → **New File** → type `hello_world.py` → **Enter**.
> In the editor that opens, type the following and save with **Ctrl+S**:
> ```
> print("Hello from VS Code + Git!")
> ```
>
> Right-click on `my-local-project` again → **New File** → type `notes.md` → **Enter**.
> Add the content below and save with **Ctrl+S**:
> ```
> # Git Commands Learned Today
>
> - git init
> - git add
> - git commit
> - git status
> - git log
> ```

#### Method 2: Code Cell (auto-create all files)

Run the cell below to create the folder and both files automatically. This is useful if you want
to skip the manual steps or verify the files have the correct content.

In [ ]:
!mkdir %USERPROFILE%\Downloads\my-local-project 2>nul
!echo print("Hello from VS Code + Git!") > "%USERPROFILE%\Downloads\my-local-project\hello_world.py"
!echo # Git Commands Learned Today > "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo. >> "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo - git init >> "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo - git add >> "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo - git commit >> "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo - git status >> "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo - git log >> "%USERPROFILE%\Downloads\my-local-project\notes.md"
!echo Files created in %USERPROFILE%\Downloads\my-local-project

### Verify your workspace tree

In [ ]:
import os

for dirname in ["git-vscode-practice", "my-local-project"]:
    folder = os.path.expanduser(f"~/Downloads/{dirname}")
    if not os.path.exists(folder):
        print(f"(not found) {dirname}/")
        continue
    print(f"├── {dirname}/")
    for entry in sorted(os.listdir(folder)):
        if entry != ".git":
            print(f"│   ├── {entry}")

### Stage and commit: three ways

Pick one method; all three produce the same result.

#### Method 1: VS Code UI (Source Control)

> 👥 **EVERYONE:** Open a **new VS Code window** (**File → New Window**), then open
> `my-local-project` as the working folder: **File → Open Folder** → navigate to
> **Downloads → my-local-project** → **Select Folder**.
> Open the **Source Control** tab (**Ctrl+Shift+G**).
> Click **Initialize Repository** (this runs `git init` for you).
> You will see `hello_world.py` and `notes.md` listed under **Changes**.
> Click the **+** next to **Changes** to stage all files.
> Type `Initial commit: add hello_world.py and notes.md` in the message box.
> Click the **✓ Commit** button.

#### Method 2: Command Palette

> 👥 **EVERYONE:** Press **Ctrl+Shift+P**.
> Type `Git: Initialize Repository` → press Enter → select the `my-local-project` folder.
> Press **Ctrl+Shift+P** again → type `Git: Stage All Changes` → Enter.
> Press **Ctrl+Shift+P** → type `Git: Commit All` → Enter.
> VS Code opens the commit message editor. Type your message, save (**Ctrl+S**), and close the tab.

#### Method 3: Terminal

Run the cell below. Switch back to this notebook window first if needed.

In [ ]:
!git -C ~/Downloads/my-local-project init
!git -C ~/Downloads/my-local-project add .
!git -C ~/Downloads/my-local-project commit -m "Initial commit: add hello_world.py and notes.md"
!git -C ~/Downloads/my-local-project log --oneline

### Check your status and history

In [ ]:
!git -C ~/Downloads/my-local-project status
!git -C ~/Downloads/my-local-project log --oneline --graph --all

## Section 5: Branching, Forking & Pull Requests (10 min)

Branches let you develop features without touching `main`.
Because the shared repo is owned by the organisation, we use the **fork workflow**,
the standard approach for open-source and cross-team collaboration on GitHub.

**The steps:**
1. **Fork** the shared repo to your personal GitHub account (one click on GitHub).
2. **Add your fork** as a remote named `fork` in your local clone.
3. **Create a personal branch** with your first name in it to avoid collisions.
4. **Add a file** named after yourself.
5. **Stage, commit, and push** to your fork.
6. **Open a Pull Request** from your fork back to `RBK-D-IT/git-vscode-practice`.


---

> ⚠️ **SECURITY: Forking and private repositories:**
> The steps below fork a **public** practice repository, so this is safe.
> **Do not apply this workflow to private org repositories** without explicit approval.
>
> When you fork a private org repo, GitHub creates a copy of the full history in your personal account.
> If you later make your personal fork public (intentionally or by accident), every commit,
> file, and secret ever stored in the org repo becomes permanently visible on the internet.
> GitHub cannot recover from this: even after reverting to private, the data may already
> be indexed or mirrored.
>
> **Rule of thumb:** If a repo has a lock icon 🔒 on GitHub, do not fork it to your personal account.
> Raise a request with your team lead instead.

---

### Step 1: Fork the repository on GitHub

> 👥 **EVERYONE:** Open a browser and go to:
> **https://github.com/RBK-D-IT/git-vscode-practice**
> Click the **Fork** button (top-right corner).
> Keep the defaults and click **Create fork**.
> GitHub redirects you to `https://github.com/<your-username>/git-vscode-practice`.
> Copy the HTTPS URL from **Code → Clone → HTTPS**. You will need it in Step 2.

### Step 2: Add your fork as a remote

**Method 1: VS Code UI (Command Palette)**

> 👥 **EVERYONE:** This must be done in the VS Code window where `git-vscode-practice` is open.
> If you cloned via the UI in Section 3, switch back to that window now.
> If you cloned via the terminal in Section 3, open the folder first:
> **File → Open Folder** → select `Downloads/git-vscode-practice` → **Select Folder**.
>
> Once `git-vscode-practice` is your active workspace, press **Ctrl+Shift+P**.
> Type `Git: Add Remote` and press Enter.
> When prompted for a **name**, type `fork` and press Enter.
> Paste your fork's HTTPS URL (copied in Step 1) and press Enter.

**Method 2: Terminal** (works from the notebook window — no window switching needed)

Edit `<your-username>` in the cell below, then run it. You should see two remotes in the output: `origin` (the org repo) and `fork` (your personal copy).

In [ ]:
# ✏️  Replace <your-username> with your GitHub username
!git -C ~/Downloads/git-vscode-practice remote add fork https://github.com/<your-username>/git-vscode-practice.git
!git -C ~/Downloads/git-vscode-practice remote -v

### Step 3: Create a personal branch

Include your first name so every attendee has a unique branch (no collisions).
Example: `feature/greeting-alex`, `feature/greeting-sam`.

**Method 1: VS Code UI (Status Bar)**

> 👥 **EVERYONE:** First, make sure your local `main` is up to date.
> Open the **Source Control** tab (**Ctrl+Shift+G**), click the **···** menu, and select **Pull**.
> Then click the **branch name** in the **bottom-left status bar** (currently shows `main`).
> Select **Create new branch**.
> Type `feature/greeting-<your-firstname>` and press **Enter**.

**Method 2: Terminal** (uses `git switch -c`, the modern alternative to `git checkout -b`)

Edit `<your-firstname>` in the cell below, then run it.

In [ ]:
# Pull latest org changes before branching
!git -C ~/Downloads/git-vscode-practice pull origin main
# ✏️  Replace <your-firstname> with your first name, e.g. feature/greeting-alex
!git -C ~/Downloads/git-vscode-practice switch -c feature/greeting-<your-firstname>


### Step 4: Add a file

Create a small markdown file named after yourself.
Everyone uses a different filename, so there are no merge conflicts when all PRs land.

**Method 1: VS Code Explorer**

> 👥 **EVERYONE:** Open the **Explorer** panel (**Ctrl+Shift+E**).
> Right-click on the `git-vscode-practice` folder → **New File**.
> Type `hello-<your-firstname>.md` and press **Enter**.
> In the editor, add the following content and save (**Ctrl+S**):
> ```
> # Hello!
>
> I completed the VS Code + Git Workflow Training.
> ```

**Method 2: Terminal**

Edit `<your-firstname>` in the cell below, then run it.

In [ ]:
# ✏️  Replace <your-firstname> with your first name, e.g. hello-alex.md
!echo # Hello! > "%USERPROFILE%\Downloads\git-vscode-practice\hello-<your-firstname>.md"
!echo. >> "%USERPROFILE%\Downloads\git-vscode-practice\hello-<your-firstname>.md"
!echo I completed the VS Code + Git Workflow Training. >> "%USERPROFILE%\Downloads\git-vscode-practice\hello-<your-firstname>.md"
!echo Created hello-<your-firstname>.md

#### Verify the file was created

Before staging, confirm Git sees the new file as **untracked** (shown in red).


In [ ]:
!git -C ~/Downloads/git-vscode-practice status


### Step 5: Stage, commit, and push to your fork

> ⚠️ **SECURITY: Verify your remote before pushing:**
> Run `git remote -v` and confirm the `fork` remote points to **your personal fork**,
> not to the org repo (`origin`).
> Pushing to the wrong remote could send your branch (and any sensitive changes) to
> a public personal fork, or overwrite branches on the org repo directly.
> If your personal fork is set to **public**, every commit you push to it is immediately
> visible to anyone on the internet, including commit messages and file contents.


**Method 1: VS Code UI (Source Control)**

> 👥 **EVERYONE:** Open the **Source Control** tab (**Ctrl+Shift+G**).
> Click **+** next to **Changes** to stage your file.
> Type `feat: add personal greeting file` in the message box.
> Click the **✓ Commit** button.
> Click **Publish Branch** (or the push icon ↑ in the status bar) to push to your fork.
> VS Code will ask which remote to publish to. Select **fork** from the list.
> VS Code will show a **"Would you like to create a Pull Request?"** notification. Click it to jump straight to Step 6.

**Method 2: Terminal**

Edit `<your-firstname>` in the cell below, then run it.

In [ ]:
!git -C ~/Downloads/git-vscode-practice add .
!git -C ~/Downloads/git-vscode-practice commit -m "feat: add personal greeting file"
# ✏️  Replace <your-firstname> to match your branch name from Step 3
!git -C ~/Downloads/git-vscode-practice push fork feature/greeting-<your-firstname>

### Step 6: Open a Pull Request

If you used **Method 1** above and clicked the VS Code notification, the PR form is already open. Skip to item 3 in the checklist below.

Otherwise (terminal push), the terminal prints a direct link. Click it, or follow these steps:

1. Go to `https://github.com/<your-username>/git-vscode-practice`
2. Click the yellow banner **"Compare & pull request"**
3. Confirm the base is `RBK-D-IT/git-vscode-practice` → `main`
4. Add a short description and click **Create pull request**

## Section 6: Summary & Cheat Sheet (5 min)

Congratulations! You have completed the hands-on portion of the session.


| Git Action | Terminal Command | VS Code UI | Command Palette (Ctrl+Shift+P) |
|---|---|---|---|
| **— SETUP —** | | | |
| Configure identity | `git config --global user.name "..."` | Accounts icon → Sign in with GitHub | — |
| Set email | `git config --global user.email "..."` | Accounts icon → Sign in with GitHub | — |
| **— INIT & CLONE —** | | | |
| Initialise a repo | `git init` | Source Control → Initialize Repository | `Git: Initialize Repository` |
| Clone a repo | `git clone <url>` | Source Control → Clone Repository | `Git: Clone` |
| **— STAGE & SNAPSHOT —** | | | |
| Check status | `git status` | Source Control tab (live badge count) | — |
| Stage a file | `git add [file]` | Click **+** next to the file | `Git: Stage Changes` |
| Stage all changes | `git add .` | Click **+** next to Changes | `Git: Stage All Changes` |
| Unstage a file | `git reset [file]` | Click **−** next to the staged file | — |
| View unstaged diff | `git diff` | Click file in Source Control panel | — |
| View staged diff | `git diff --staged` | Click staged file in Source Control | — |
| Commit staged files | `git commit -m "msg"` | Type message → click ✓ Commit | `Git: Commit` |
| **— BRANCH & MERGE —** | | | |
| List branches | `git branch` | Click branch name in status bar | — |
| Create a branch | `git switch -c <name>` | Click branch name → Create new branch | `Git: Create Branch` |
| Switch branch | `git switch <name>` | Click branch name in status bar | `Git: Checkout to...` |
| Merge a branch | `git merge <branch>` | — | `Git: Merge Branch` |
| **— HISTORY & COMPARE —** | | | |
| View history | `git log --oneline` | Timeline panel (Explorer tab) | `Git: Show History` |
| Visual branch graph | `git log --oneline --graph --all` | — | — |
| Commits on A not on B | `git log branchB..branchA` | — | — |
| Diff between branches | `git diff branchB...branchA` | — | — |
| Show a commit | `git show [SHA]` | Click commit in Timeline panel | — |
| **— REMOTES & SYNC —** | | | |
| Add a remote | `git remote add <name> <url>` | — | `Git: Add Remote` |
| Fetch from remote | `git fetch` | Source Control → ··· → Fetch | `Git: Fetch` |
| Push a branch | `git push <remote> <branch>` | Source Control → Publish Branch | `Git: Push` |
| Pull latest | `git pull` | Source Control → Pull | `Git: Pull` |
| **— FORK & PR —** | | | |
| Fork a repo | GitHub web UI only | — | — |
| Open a Pull Request | Click link printed after push | VS Code PR notification | — |
| **— STASH —** | | | |
| Save work temporarily | `git stash` | Source Control → ··· → Stash All | — |
| List stashes | `git stash list` | — | — |
| Apply latest stash | `git stash pop` | Source Control → ··· → Pop Stash | — |
| Discard latest stash | `git stash drop` | — | — |
| **— FILE OPERATIONS —** | | | |
| Delete a tracked file | `git rm [file]` | Explorer → right-click → Delete | — |
| Move / rename a file | `git mv [old] [new]` | Explorer → right-click → Rename | — |
| Ignore files | Add patterns to `.gitignore` | Explorer → New File → `.gitignore` | — |

---

## 🎉 That's a wrap on the hands-on section!

The floor is now open for **20 minutes of Q&A**.
Bring any questions you have about Git, VS Code, branching strategies,
or how these workflows apply to your day-to-day data work.